In [1]:
import csv
import json
import os
import re
import pandas as pd
import prosodic

In [2]:
with open("hopkins.txt", "r", encoding="utf-8") as file:
    book = file.read()

# took the following from word cloud example in week 10
start = book.index("*** START OF")
end = book.index("*** END OF")
text = book[start:end]

In [3]:
titles = re.findall(r'_(\d+.*?)_', text, re.DOTALL) #googled some regex for this
print(len(titles))

71


In [4]:
print(titles[0])

1
For a Picture of
St. Dorothea


In [5]:
poems = []
for i in range(70):
    poem_start = text.index(titles[i]) #taken from prof's example in week 10
    poem_end = text.index(titles[i+1])
    if poem_end is not None: #taken from prof's work on my first try
        poem = text[poem_start:poem_end]
    else: 
        poem = text[poem_start:end]
    pt = poem.replace("_\n\n", "")
    pt = pt.strip("\n\n\n")
    pt = pt.strip("\n\n\n_")
    poem_object = {
        "title": titles[i],
        "poem_text": pt.replace(titles[i], "")
    }
    poems.append(poem_object) 

In [6]:
print(poems[0]) #checking it worked

{'title': '1\nFor a Picture of\nSt. Dorothea', 'poem_text': "I BEAR a basket lined with grass;\nI am so light, I am so fair,\nThat men must wonder as I pass\nAnd at the basket that I bear,\nWhere in a newly-drawn green litter\nSweet flowers I carry,--sweets for bitter.\n\nLilies I shew you, lilies none,\nNone in Caesar's gardens blow,--\nAnd a quince in hand,--not one\nIs set upon your boughs below;\nNot set, because their buds not spring;\nSpring not, 'cause world is wintering.\n\nBut these were found in the East and South\nWhere Winter is the clime forgot.--\nThe dewdrop on the larkspur's mouth\nO should it then be quenchèd not?\nIn starry water-meads they drew\nThese drops: which be they? stars or dew?\n\nHad she a quince in hand? Yet gaze:\nRather it is the sizing moon.\nLo, linked heavens with milky ways!\nThat was her larkspur row.--So soon?\nSphered so fast, sweet soul?--We see\nNor fruit, nor flowers, nor Dorothy."}


In [7]:
df = pd.DataFrame(poems) #creating a dataframe to make it easier to navigate

In [8]:
df.head()

,title,poem_text
0,1\nFor a Picture of\nSt. Dorothea,I BEAR a basket lined with grass;\nI am so lig...
1,2\nHeaven--Haven\nA nun takes the veil,I HAVE desired to go\n Where springs no...
2,3\nThe Habit of Perfection,"ELECTED Silence, sing to me\nAnd beat upon my ..."
3,4\nTHE WRECK\nOF THE DEUTSCHLAND,To the\nhappy memory of five Francis...
4,5\nPenmaen Pool,_For the Visitors' Book at the InnWHO long for...


In [9]:
print(df["poem_text"][0])

I BEAR a basket lined with grass;
I am so light, I am so fair,
That men must wonder as I pass
And at the basket that I bear,
Where in a newly-drawn green litter
Sweet flowers I carry,--sweets for bitter.

Lilies I shew you, lilies none,
None in Caesar's gardens blow,--
And a quince in hand,--not one
Is set upon your boughs below;
Not set, because their buds not spring;
Spring not, 'cause world is wintering.

But these were found in the East and South
Where Winter is the clime forgot.--
The dewdrop on the larkspur's mouth
O should it then be quenchèd not?
In starry water-meads they drew
These drops: which be they? stars or dew?

Had she a quince in hand? Yet gaze:
Rather it is the sizing moon.
Lo, linked heavens with milky ways!
That was her larkspur row.--So soon?
Sphered so fast, sweet soul?--We see
Nor fruit, nor flowers, nor Dorothy.


In [10]:
#p_poem = prosodic.Text(df["poem_text"][0])
#p_poem.parse()
#for l in p_poem.lines:
   #print(l.best_parse.txt)

In [11]:
all_parsed_poems = []
valid_poems = df["poem_text"].dropna() #I was getting an error saying the string can't evaluate to none, so I'm dropping null values
valid_poems = valid_poems[valid_poems.str.strip().astype(bool)] #google said to add this extra step removing empty strings: "" 
for vp in valid_poems[:69]:
    if vp is not None:
        parsable_poem = prosodic.Text(vp)
        parsed = parsable_poem.parse()
        # add some lines to compute the integers for the counts 
        poem_info = {
            "feet_per_line" : parsable_poem.line_scheme, #this is a dict
            "syll_per_line" : parsable_poem.syllable_scheme, #this is a dict
            "line_count" : len(parsable_poem.lines),
            "stanza_count" : len(parsable_poem.stanzas),
            "parsed_text" : parsable_poem.Text.txt,
            "summary" : parsable_poem.summary()
            }
        all_parsed_poems.append(poem_info)


[4.27s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls); falling back to linepart parsing
[0.04s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls)
[0.24s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls); falling back to linepart parsing
[1.43s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls)
[9.86s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls); falling back to linepart parsing
[0.62s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls)
[8.65s] prosodic.parsing.vectorized.parse_batch_from_df(): 9 line(s) exceed MAX_SYLL_IN_

In [47]:
parsed_df = pd.DataFrame(all_parsed_poems)

In [79]:
all_parsed_lines = []
for vp in valid_poems[:69]:
    if vp is not None:
        parsable_poem = prosodic.Text(vp)
        parsable_poem.parse() 
        if parsable_poem.lines is not None:
            for l in parsable_poem.lines:
                  if l is not None and getattr(l, "best_parse", None) is not None and l.best_parse.feet is not None:
                      line_info = {
                           "feet_in_line": len(l.best_parse.feet),
                          "parsed_text": l.txt
                    }
                      all_parsed_lines.append(line_info)

[154.43s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls); falling back to linepart parsing
[0.04s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=22 sylls)
[0.09s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls); falling back to linepart parsing
[1.29s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=21 sylls)
[3.31s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 line(s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls); falling back to linepart parsing
[0.61s] prosodic.parsing.vectorized.parse_batch_from_df(): 1 (s) exceed MAX_SYLL_IN_PARSE_UNIT=18 and were not parsed (max=25 sylls)
[6.21s] prosodic.parsing.vectorized.parse_batch_from_df(): 9 line(s) exceed MAX_SYLL_I

In [80]:
parsed_line_df = pd.DataFrame(all_parsed_lines)
parsed_line_df

,feet_in_line,parsed_text
0,4,I BEAR a basket lined with grass;
1,4,"\nI am so light, I am so fair,"
2,4,\nThat men must wonder as I pass
3,4,"\nAnd at the basket that I bear,"
4,5,\nWhere in a newly-drawn green litter
...,...,...
3197,4,\nAnd then that last and shortest . . .
3198,5,"STRIKE, churl; hurl, cheerless wind, then; hel..."
3199,6,\nMay's beauty massacre and wispèd wild clouds...
3200,5,"\nOut on the giant air; tell Summer No,"


### Issues to note in final summary
- The parsing doesn't work with any white space
- The parsing has a limit of 18 syllables per line, and some lines are longer, so it falls back to a "line-part"

### Ideas - parsing poems
- Manually select a few poems as case studies (early career, mid-career, late career / at least one sprung rhythm example / one with white space?) -- maybe just for visual examples.
  
### Ideas - analysis
- count line length per poem over time and on average
- count feet per line over time and on average
- count the most common patterns (iambic, trochaic, spondaic, dactyllic, anapestic)
- *challenge*: try to find an agreed-upon definition of sprung rhythm and see if the code identifies a matching meter, or use the custom meter function in prosodic to find sprung rhythm poems!